In [1]:
import pandas as pd
import numpy as np
from gensim.models.phrases import Phrases, Phraser
from gensim.models import Word2Vec
from gensim.test.utils import get_tmpfile
from gensim.models import KeyedVectors
from time import time 
import multiprocessing
from tqdm import tqdm
from re import sub
import re
tqdm.pandas()

C:\Libraries\anaconda3\lib\site-packages\tqdm\std.py:697: FutureWarning: The Panel class is removed from pandas. Accessing it from the top-level namespace will also be removed in the next version
  from pandas import Panel


In [2]:
df = pd.read_csv("./sentiment140-cleaned.csv", encoding="ISO-8859-1", engine="python")



In [3]:
df['text'] = df.progress_apply(lambda row: row['text'].split(), axis=1)

100%|██████████| 1600000/1600000 [00:23<00:00, 68982.75it/s] 


In [4]:
print(df.head())

   sentiment             user  \
0          0  _TheSpecialOne_   
1          0    scotthamilton   
2          0         mattycus   
3          0          ElleCTF   
4          0           Karoli   

                                                text  
0  [@switchfoot, awww, that, is, a, bummer, you, ...  
1  [is, upset, that, he, can, t, update, his, fac...  
2  [@kenichan, i, dived, many, times, for, the, b...  
3  [my, whole, body, feels, itchy, and, like, its...  
4  [@nationwideclass, no, it, is, not, behaving, ...  


In [5]:
sent = [row for row in df['text']]
phrases = Phrases(sent, min_count=1, progress_per=50000)
bigram = Phraser(phrases)
sentences = bigram[sent]
sentences[1]

['is',
 'upset',
 'that',
 'he',
 'can_t',
 'update',
 'his',
 'facebook',
 'by',
 'texting',
 'it',
 'and',
 'might',
 'cry',
 'as',
 'a',
 'result',
 'school',
 'today',
 'also',
 'blah',
 '!']

In [16]:
w2v_model = Word2Vec(
    min_count=3,
    window=4,
    size=300,
    sample=1e-5,
    alpha=0.03,
    min_alpha=0.0007,
    negative=20,
    workers=multiprocessing.cpu_count()-1)

In [17]:
start = time()
w2v_model.build_vocab([x for x in tqdm(sentences)], progress_per=50000)
print('Time to build vocab: {} mins'.format(round((time() - start) / 60, 2)))

100%|██████████| 1600000/1600000 [00:50<00:00, 31852.56it/s]
Time to build vocab: 1.41 mins


In [18]:
start = time()

w2v_model.train([x for x in tqdm(sentences)], total_examples=w2v_model.corpus_count, epochs=30, report_delay=1)

print('Time to train the model: {} mins'.format(round((time() - start) / 60, 2)))

w2v_model.init_sims(replace=True)

100%|██████████| 1600000/1600000 [00:49<00:00, 32167.69it/s]
Time to train the model: 7.49 mins


In [20]:
w2v_model.save("./sentiment140-word2vec_defaults.model")